In [ ]:
%pip install pandas mp-api


In [ ]:
import pandas as pd
from mp_api.client import MPRester
from itertools import combinations

API_KEY = "G9LvXEqC4OTlP7YMrQNHmW3pyc3ip7C8"

# Combine all elements to pick combinations from
elements = ["Al", "Ga", "In", "N", "P", "As", "Sb"]

# Generate all possible combinations of 2, 3, and 4 elements
# This covers binary, ternary, and quaternary systems
iii_v_systems = []
for r in range(2, 5):  # r=2 (binary), r=3 (ternary), r=4 (quaternary)
    for combo in combinations(elements, r):
        # Join elements with a hyphen for the API query
        iii_v_systems.append("-".join(combo))

print(f"Searching for {len(iii_v_systems)} chemical systems.")

with MPRester(API_KEY) as mpr:
    # Use the expanded list of chemical systems
    results = mpr.materials.summary.search(
        chemsys=iii_v_systems,
        fields=["material_id", "formula_pretty", "band_gap", "volume", "density", "symmetry"]
    )

print(f"Successfully downloaded {len(results)} III-V crystal structures!")

In [ ]:
data_list = []
for doc in results:
    data_list.append({
        "material_id": doc.material_id,
        "formula": doc.formula_pretty,
        "band_gap": doc.band_gap,
        "volume": doc.volume,
        "density": doc.density,
        "crystal_system": doc.symmetry.crystal_system.name if doc.symmetry else None
    })

# 2. Create the DataFrame
full_df = pd.DataFrame(data_list)

# 3. Now you have full_df! Let's clean it (remove metallic/None values)
# Remove entries where band_gap is None or 0.0
full_df = full_df[full_df['band_gap'].notna() & (full_df['band_gap'] > 0)]

# 4. Show the first few rows
print(f"Final dataset shape: {full_df.shape}")
display(full_df.head())

In [ ]:
import pandas as pd

# 1. Grab the list of all semiconductor material IDs from your current DataFrame
material_ids = full_df['material_id'].tolist()

# 2. Create an empty list to store our geometric data
lattice_data = []

# 3. Query the Materials Project database for the exact structures
with MPRester("G9LvXEqC4OTlP7YMrQNHmW3pyc3ip7C8") as mpr:
    # We request the 'structure' field which contains the full 3D lattice object
    docs = mpr.summary.search(material_ids=material_ids, fields=["material_id", "structure"])
    
    # 4. Loop through the downloaded documents and extract all 6 parameters
    for doc in docs:
        lattice = doc.structure.lattice
        lattice_data.append({
            "material_id": doc.material_id,
            "Lattice_a": lattice.a,
            "Lattice_b": lattice.b,
            "Lattice_c": lattice.c,
            "Alpha": lattice.alpha,
            "Beta": lattice.beta,
            "Gamma": lattice.gamma
        })

# 5. Convert this new geometric data into its own temporary DataFrame
lattice_df = pd.DataFrame(lattice_data)

# 6. Merge the new 3D parameters back into your main dataset
comprehensive_df = pd.merge(full_df, lattice_df, on="material_id")

# 7. View the incredibly detailed, comprehensive dataset
# 1. Force Pandas to show all columns (removes the ... in the middle)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# 2. Display the entire, unaltered comprehensive dataset
display(comprehensive_df)




In [ ]:
import pandas as pd

# Show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Now printing the DataFrame will show everything
print(comprehensive_df)


In [ ]:
comprehensive_df.to_csv("III_V_Semiconductors_Datasetall.csv", index=False)